# BPLLM trên Kaggle (1/2): chạy thực nghiệm và chấm điểm (oracle)

Tái lập Bảng 4 của bài báo: model Llama 3.1 8B đã fine-tune của tác giả, prompt Hình 4,
top-10 chunk BPMN, so chunk **specific** (như bài báo) với **enriched** (phần mở rộng).
Model nạp ở `float16`, không lượng tử hóa lúc chạy, đúng trọng số trên Hugging Face.

**Cài đặt ở cột bên phải trước khi chạy:**
1. **Accelerator:** GPU T4 x2
2. **Internet:** On
3. **Input:** Add Input, chọn Kaggle dataset chứa các file `data/rag_inputs/*_top10.jsonl`
   do `python scripts/retrieve_top_k.py` sinh trên máy local (xem `docs/huong_dan_chay.md`)

Chạy lần lượt từ trên xuống. Đổi thực nghiệm hoặc model: sửa Khối 1 rồi chạy lại từ Khối 1.
Mỗi model tải khoảng 16 GB; khối nạp model tự xóa model cũ khỏi bộ nhớ và ổ đĩa.

## Khối 1. Cấu hình

| Biến | Giá trị | Nghĩa |
|---|---|---|
| `EXPERIMENT` | `fd_refined`, `fd_ft`, `similar`, `different` | Bộ câu hỏi 3, 6, 5, 4 (xem `src/bpllm/experiments.py`) |
| `QUANT` | `int4`, `int8` | Bản model của tác giả (fine-tune PEFT int4 hoặc int8) |
| `CHUNKINGS` | `["specific", "enriched"]` | Chạy cả hai để so sánh |
| `PROMPT_STYLE` | `chat`, `raw` | `chat`: qua chat template Llama 3.1 (như model card, mặc định). `raw`: văn bản thô + `Helpful Answer:` |

Nhóm model đi theo thực nghiệm: `fd_*` dùng model Food Delivery, `similar` dùng model
Food Delivery + E-commerce, `different` dùng model Food Delivery + Reimbursement.

In [ ]:
EXPERIMENT = "fd_ft"
QUANT = "int4"
CHUNKINGS = ["specific", "enriched"]
PROMPT_STYLE = "chat"
MAX_NEW_TOKENS = 128
TOP_K = 10

# Bản chép của src/bpllm/experiments.py (tests/test_notebooks.py kiểm tra hai nơi khớp nhau)
MODELS = {
    "food_delivery": {"int4": "angeloc1/llama3dot1FoodDel4v05",
                      "int8": "angeloc1/llama3dot1FoodDel8v02"},
    "similar": {"int4": "angeloc1/llama3dot1SimilarProcesses4",
                "int8": "angeloc1/llama3dot1SimilarProcesses8"},
    "different": {"int4": "angeloc1/llama3dot1DifferentProcesses4",
                  "int8": "angeloc1/llama3dot1DifferentProcesses8"},
}
EXPERIMENTS = {
    "fd_refined": {"model_group": "food_delivery", "count": 104},
    "fd_ft": {"model_group": "food_delivery", "count": 104},
    "similar": {"model_group": "similar", "count": 190},
    "different": {"model_group": "different", "count": 182},
}

assert EXPERIMENT in EXPERIMENTS, f"EXPERIMENT phải là một trong {list(EXPERIMENTS)}"
assert QUANT in ("int4", "int8"), "QUANT phải là int4 hoặc int8"
assert set(CHUNKINGS) <= {"specific", "enriched"}, "CHUNKINGS chỉ gồm specific, enriched"
assert PROMPT_STYLE in ("chat", "raw"), "PROMPT_STYLE phải là chat hoặc raw"

MODEL_PATH = MODELS[EXPERIMENTS[EXPERIMENT]["model_group"]][QUANT]
print("Thực nghiệm:", EXPERIMENT, "| model:", MODEL_PATH, "| chunk:", CHUNKINGS, "| prompt:", PROMPT_STYLE)

## Khối 2. Kiểm tra GPU

Phải thấy 2 Tesla T4. Nếu báo lỗi thì chưa bật Accelerator.

In [ ]:
!nvidia-smi

## Khối 3. Cài thư viện

Chỉ cài gói còn thiếu; không hạ cấp thư viện Kaggle cài sẵn.

In [ ]:
import importlib.metadata as md
import subprocess
import sys

from packaging.version import Version

MISSING = []


def need(package, minimum):
  try:
    current = md.version(package)
  except md.PackageNotFoundError:
    current = None
  if current is None or (minimum and Version(current) < Version(minimum)):
    MISSING.append(f"{package}>={minimum}" if minimum else package)
  return current


need("transformers", "4.45")
need("accelerate", "0.33")
need("bitsandbytes", "0.43")

if MISSING:
  print("Cài:", MISSING)
  subprocess.run([sys.executable, "-m", "pip", "install", "-q", *MISSING], check=True)
  print("Đã cài xong. Nếu vừa nâng cấp transformers: Run → Restart & clear cell outputs, rồi chạy lại từ Khối 1.")
for package in ("torch", "transformers", "huggingface_hub", "accelerate", "bitsandbytes"):
  try:
    print(f"{package:16} {md.version(package)}")
  except md.PackageNotFoundError:
    print(f"{package:16} (chưa có)")

## Khối 4. Tìm file input

Tên file: `<thực nghiệm>_<chunk>_top10.jsonl`. Notebook gửi nguyên văn trường `prompt`
(dựng sẵn bởi `src/bpllm/rag/prompts.py`), nên prompt lúc chấm và lúc chạy app là một.

In [ ]:
import glob
import json
import os

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"


def find_input(name):
  matches = glob.glob(f"{INPUT_DIR}/**/{name}", recursive=True)
  assert matches, f"Không thấy {name} trong {INPUT_DIR}. Hãy Add Input dataset chứa file này."
  return matches[0]


def read_jsonl(path):
  with open(path, encoding="utf-8") as f:
    return [json.loads(line) for line in f if line.strip()]


INPUT_FILES = {c: find_input(f"{EXPERIMENT}_{c}_top{TOP_K}.jsonl") for c in CHUNKINGS}
for chunking, path in INPUT_FILES.items():
  records = read_jsonl(path)
  assert len(records) == EXPERIMENTS[EXPERIMENT]["count"], f"{path}: {len(records)} câu"
  assert all(r["chunking"] == chunking for r in records), f"{path} không phải chunk {chunking}"
  print(f"{chunking}: {len(records)} câu, kho {records[0]['processes']}, {path}")

## Khối 5a. Chuẩn bị tải model

Đặt HF token (không bắt buộc) ở **Add-ons → Secrets**, tên `HF_TOKEN` (token đọc của
Hugging Face), để tránh bị giới hạn khi tải ẩn danh 16 GB.

In [ ]:
import os
import shutil
import time

import torch
import transformers
from huggingface_hub import snapshot_download
from packaging.version import Version

HF_TOKEN = None
try:
  from kaggle_secrets import UserSecretsClient
  HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
  pass
print("HF token:", "có (Kaggle Secret HF_TOKEN)" if HF_TOKEN else "không (tải ẩn danh, có thể bị giới hạn tốc độ)")

# transformers >= 4.56 đổi torch_dtype thành dtype
DTYPE_KW = ({"dtype": torch.float16} if Version(transformers.__version__) >= Version("4.56")
            else {"torch_dtype": torch.float16})


def free_gb(path):
  return shutil.disk_usage(path).free / 2**30


# Thư mục tải model: chọn ổ còn nhiều chỗ nhất (mỗi model cần khoảng 16,1 GB)
MODEL_ROOT = max(["/tmp", "/kaggle/working"], key=free_gb)
MODEL_ROOT = os.path.join(MODEL_ROOT, "bpllm_models")
os.makedirs(MODEL_ROOT, exist_ok=True)
print(f"Tải model vào {MODEL_ROOT}, còn trống {free_gb(MODEL_ROOT):.1f} GB")


def download(model_id):
  """Tải trọng số + tokenizer (bỏ training_args.bin); trả về thư mục local. Lỗi thì nói rõ nguyên nhân."""
  target = os.path.join(MODEL_ROOT, model_id.replace("/", "__"))
  if free_gb(MODEL_ROOT) < 17:
    raise RuntimeError(f"Ổ {MODEL_ROOT} chỉ còn {free_gb(MODEL_ROOT):.1f} GB, cần ~17 GB. "
                       "Xóa model cũ trong thư mục này hoặc Restart session.")
  start = time.time()
  for attempt in range(1, 4):
    try:
      path = snapshot_download(model_id, local_dir=target, token=HF_TOKEN, max_workers=8,
                               allow_patterns=["*.json", "*.safetensors"])
      break
    except Exception as error:
      print(f"  Lần {attempt} tải {model_id} lỗi: {type(error).__name__}: {error}")
      if attempt == 3:
        raise RuntimeError(
            "Không tải được model. Kiểm tra: (1) Settings → Internet: On; "
            "(2) đã xác minh số điện thoại Kaggle; (3) thêm Kaggle Secret HF_TOKEN nếu lỗi 429/401; "
            "(4) dung lượng ổ ở dòng in phía trên.") from error
      time.sleep(15 * attempt)
  shards = [f for f in os.listdir(path) if f.endswith(".safetensors")]
  print(f"  Đã tải {model_id}: {len(shards)} file safetensors, {time.time() - start:.0f} s")
  return path


def remove_download(path):
  shutil.rmtree(path, ignore_errors=True)

## Khối 5b. Tải và nạp model (float16, chia đều 2 GPU)

Nếu đang có model khác (vừa đổi Khối 1), khối này giải phóng GPU và xóa bản tải của model cũ
để ổ đĩa Kaggle không đầy.

In [ ]:
import gc

from transformers import AutoModelForCausalLM, AutoTokenizer

if globals().get("LOADED_MODEL") not in (None, MODEL_PATH):
  del model
  gc.collect()
  torch.cuda.empty_cache()
  remove_download(LOADED_DIR)
  print("Đã xóa model cũ:", LOADED_MODEL)

if globals().get("LOADED_MODEL") != MODEL_PATH:
  LOADED_DIR = download(MODEL_PATH)
  tokenizer = AutoTokenizer.from_pretrained(LOADED_DIR)
  model = AutoModelForCausalLM.from_pretrained(LOADED_DIR, device_map="auto", **DTYPE_KW).eval()
  LOADED_MODEL = MODEL_PATH
print("Đã nạp:", LOADED_MODEL)

## Khối 6. Hàm hỏi model

Mỗi câu một lần hỏi độc lập, giải mã greedy; model không nhận `expected`.

In [ ]:
def ask(prompt):
  assert LOADED_MODEL == MODEL_PATH, "Đã đổi cấu hình: chạy lại Khối 5."
  if PROMPT_STYLE == "chat":
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors="pt")
  else:
    inputs = tokenizer(prompt + "\nHelpful Answer:", return_tensors="pt")
  inputs = inputs.to(model.device)
  with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                temperature=None, top_p=None,
                                pad_token_id=tokenizer.eos_token_id)
  text = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
  if PROMPT_STYLE == "raw":
    text = text.split("\nQuestion:")[0]
  return text.strip()

## Khối 7. Thử 1 câu hỏi

In [ ]:
sample = read_jsonl(INPUT_FILES[CHUNKINGS[0]])[0]
print(sample["prompt"])
print("-" * 70)
print("Model trả lời:", ask(sample["prompt"]))
print("Đáp án đúng  :", sample["expected"])

## Khối 8. Chạy toàn bộ câu hỏi

Ghi `/kaggle/working/answers_<thực nghiệm>_<chunk>_<quant>.jsonl`. Nếu phiên Kaggle bị ngắt,
chạy lại khối này: câu đã có trong file được bỏ qua, chạy tiếp từ câu còn thiếu.

In [ ]:
from tqdm.auto import tqdm


def answers_file(chunking):
  return f"{OUTPUT_DIR}/answers_{EXPERIMENT}_{chunking}_{QUANT}.jsonl"


def run(chunking):
  path = answers_file(chunking)
  done = {r["id"] for r in read_jsonl(path)} if os.path.exists(path) else set()
  with open(path, "a", encoding="utf-8") as f:
    for record in tqdm(read_jsonl(INPUT_FILES[chunking]), desc=f"{EXPERIMENT} {chunking} {QUANT}"):
      if record["id"] in done:
        continue
      row = {"id": record["id"], "question": record["question"], "expected": record["expected"],
             "answer": ask(record["prompt"]), "model": MODEL_PATH, "experiment": EXPERIMENT,
             "chunking": chunking, "quant": QUANT, "prompt_style": PROMPT_STYLE}
      f.write(json.dumps(row, ensure_ascii=False) + "\n")
      f.flush()
  return path


ANSWER_FILES = {chunking: run(chunking) for chunking in CHUNKINGS}

## Khối 9. Oracle

Bản chép của `src/bpllm/evaluation/oracle.py` (test kiểm tra hai nơi khớp nhau).

In [ ]:
import re

_YES_NO = re.compile(r"\b(yes|no)\b", re.IGNORECASE)
LABELS = ("yes", "no")


def extract_answer(text):
  match = _YES_NO.search(text or "")
  return match.group(1).lower() if match else None


def grade(records):
  total = {label: 0 for label in LABELS}
  correct = {label: 0 for label in LABELS}
  wrong, unparsed = [], []
  for i, record in enumerate(records, 1):
    qid = record.get("id", i)
    expected = str(record["expected"]).strip().lower()
    if expected not in LABELS:
      raise ValueError(f"câu {qid}: expected phải là yes/no, nhận được '{record['expected']}'")
    predicted = extract_answer(record.get("answer"))
    total[expected] += 1
    if predicted is None:
      unparsed.append(qid)
    if predicted == expected:
      correct[expected] += 1
    else:
      wrong.append(qid)
  n = len(records)
  return {
      "total": n,
      "correct": sum(correct.values()),
      "accuracy": sum(correct.values()) / n if n else 0.0,
      "accuracy_yes": correct["yes"] / total["yes"] if total["yes"] else 0.0,
      "accuracy_no": correct["no"] / total["no"] if total["no"] else 0.0,
      "wrong_ids": wrong,
      "unparsed_ids": unparsed,
  }

## Khối 10. Chấm điểm, in câu sai, lưu báo cáo

In [ ]:
REPORTS = {}
for chunking, path in ANSWER_FILES.items():
  records = read_jsonl(path)
  report = grade(records)
  REPORTS[chunking] = report
  print(f"[{EXPERIMENT} / {chunking} / {QUANT}] accuracy {report['accuracy']:.2%} "
        f"({report['correct']}/{report['total']}), yes {report['accuracy_yes']:.2%}, "
        f"no {report['accuracy_no']:.2%}, không rút được: {report['unparsed_ids']}")
  for r in records:
    if r["id"] in report["wrong_ids"]:
      print(f"  Sai câu {r['id']} (đúng là {r['expected']}): {r['question'][:120]}")
      print(f"      Model: {r['answer'][:200]}")

report_file = f"{OUTPUT_DIR}/report_{EXPERIMENT}_{QUANT}.json"
with open(report_file, "w", encoding="utf-8") as f:
  json.dump({"experiment": EXPERIMENT, "model": MODEL_PATH, "quant": QUANT,
             "prompt_style": PROMPT_STYLE, "max_new_tokens": MAX_NEW_TOKENS,
             "reports": REPORTS}, f, indent=2, ensure_ascii=False)
print("Đã lưu:", report_file)

## Khối 11. Đưa kết quả về repo

Ở cột bên phải, mục **Output** (`/kaggle/working`), tải các file `answers_*.jsonl` về
`data/results/` của repo, rồi trên máy local chạy:

```
python scripts/evaluate.py
```

để lập bảng tổng hợp `data/results/summary.md` cho mọi thực nghiệm đã chạy.